# 🐧 Linux User & Group Management

---

## 🏠 The Big Picture: Think of Linux Like an Apartment Building

| Real World | Linux Equivalent |
|------------|-----------------|
| Apartment Building | The Linux System |
| Residents | Users |
| Families/Roommates | Groups |
| Apartment Keys | Permissions |
| Building Manager | Root (superuser) |
| Tenant List | `/etc/passwd` |
| Master Key Log | `/etc/shadow` |
| Family Register | `/etc/group` |

---

## 👥 1. Creating & Deleting Users/Groups

### 🆕 Creating a User (Adding a New Resident)
```bash
# Basic way - like giving someone an apartment
sudo useradd alice

# Better way - with home folder & default settings
sudo useradd -m -s /bin/bash alice

# With extra details (like a welcome package!)
sudo useradd -m -s /bin/bash -c "Alice Developer" -G developers,sudo alice
```

### 🗑️ Deleting a User (When Someone Moves Out)
```bash
# Remove user but keep their files (like leaving furniture behind)
sudo userdel alice

# Remove user AND their home folder (clean sweep!)
sudo userdel -r alice
```

### 👨‍👩‍👧 Creating & Managing Groups (Families/Teams)
```bash
# Create a new group (like forming a club)
sudo groupadd developers

# Add user to a group (invite them to the club)
sudo usermod -aG developers alice

# Remove user from a group
sudo gpasswd -d alice developers

# Delete a group (dissolve the club)
sudo groupdel developers
```

> 💡 **Pro Tip**: Always use `-aG` (append to groups) with `usermod`, NOT `-G` alone — otherwise you'll remove them from ALL other groups! 😱

---

## 🛠️ 2. The 3 Magic Commands: `useradd`, `userdel`, `usermod`

| Command | What It Does | Real-Life Analogy |
|---------|-------------|-------------------|
| `useradd` | Creates a new user | Signing a new lease |
| `userdel` | Deletes a user | Ending a lease & moving out |
| `usermod` | Modifies existing user | Renovating or changing apartment details |

### 🔧 Common `usermod` Options (Your Toolkit)
```bash
# Change username
sudo usermod -l newname oldname

# Change user's home directory
sudo usermod -d /new/home -m username

# Add to supplementary group (MOST IMPORTANT!)
sudo usermod -aG sudo username    # Give admin powers
sudo usermod -aG docker username  # Allow Docker access

# Lock/unlock account
sudo usermod -L username   # Lock (like changing locks)
sudo usermod -U username   # Unlock
```

---

## 📁 3. The 3 Important Files (Linux's "Tenant Database")

### 🔹 `/etc/passwd` — The Public Directory
```
alice:x:1001:1001:Alice Developer:/home/alice:/bin/bash
```
| Field | Meaning | Example |
|-------|---------|---------|
| 1 | Username | `alice` |
| 2 | Password placeholder | `x` (real password is in `/etc/shadow`) |
| 3 | User ID (UID) | `1001` |
| 4 | Group ID (GID) | `1001` |
| 5 | User info (GECOS) | `Alice Developer` |
| 6 | Home directory | `/home/alice` |
| 7 | Default shell | `/bin/bash` |

> ✅ **Anyone can read this** — it's like a public phone directory.

---

### 🔹 `/etc/shadow` — The Secret Vault 🔐
```
alice:$6$xyz...:19000:0:99999:7:::
```
| Field | Meaning |
|-------|---------|
| 1 | Username |
| 2 | Encrypted password (or `!` if locked, `*` if no login) |
| 3 | Days since Jan 1, 1970 when password was last changed |
| 4 | Minimum days before password can be changed |
| 5 | Maximum days password is valid |
| 6 | Days before expiry to warn user |
| 7 | Days after expiry before account is disabled |

> 🔒 **Only root can read this** — like a bank vault for passwords!

---

### 🔹 `/etc/group` — The Group Register
```
developers:x:1002:alice,bob,charlie
```
| Field | Meaning |
|-------|---------|
| 1 | Group name |
| 2 | Password placeholder (rarely used) |
| 3 | Group ID (GID) |
| 4 | List of users in this group |

> 👥 This tells Linux: *"These people belong to this team!"*

---

## 🔐 4. Password Policies (Making Passwords Strong & Safe)

### 🛡️ Why Policies Matter
> Just like your apartment building has rules:  
> *"No copying keys"*, *"Change locks every 90 days"*, *"Minimum 6-digit code"*

### ⚙️ How to Set Password Policies

#### For a single user:
```bash
# Force password change on next login
sudo chage -d 0 alice

# Set password to expire in 90 days
sudo chage -M 90 alice

# Warn 7 days before expiration
sudo chage -W 7 alice

# Lock account after password expires + 14 days
sudo chage -E $(($(date +%s)/86400 + 14)) alice
```

#### System-wide policy (`/etc/login.defs`):
```bash
# Edit this file to set defaults for NEW users
PASS_MAX_DAYS   90    # Max password age
PASS_MIN_DAYS   1     # Min days between changes
PASS_WARN_AGE   7     # Warning days before expiry
```

#### Enforce strong passwords (`/etc/pam.d/common-password`):
```bash
# Example: Require 12 chars, 1 uppercase, 1 digit, 1 special char
password requisite pam_pwquality.so retry=3 minlen=12 ucredit=-1 dcredit=-1 ocredit=-1
```

> 💡 **Teach this**: *"Good password policy = locking your digital front door!"*

---

## 🎯 5. Practice: Create Users with Different Privilege Levels

Let's build a mini-company structure! 🏢

### Step 1: Create Groups (Departments)
```bash
sudo groupadd admins      # IT Managers
sudo groupadd developers  # Dev Team
sudo groupadd interns     # Trainees
```

### Step 2: Create Users with Different Access

```bash
# 👑 Admin: Full system access
sudo useradd -m -s /bin/bash -c "Sarah Admin" -G admins,sudo sarah
sudo passwd sarah

# 👨‍💻 Developer: Can code, deploy, but NOT break the system
sudo useradd -m -s /bin/bash -c "Tom Dev" -G developers tom
sudo passwd tom
# Optional: Allow specific sudo commands only
echo "tom ALL=(ALL) NOPASSWD: /usr/bin/systemctl restart nginx" | sudo tee /etc/sudoers.d/tom

# 👶 Intern: Very limited access
sudo useradd -m -s /bin/rbash -c "Alex Intern" -G interns alex
sudo passwd alex
# Restrict what they can do (restricted bash)
```

### Step 3: Verify & Test
```bash
# Check user details
id sarah
# Output: uid=1003(sarah) gid=1003(sarah) groups=1003(sarah),27(sudo),1000(admins)

# Switch to user and test
sudo -i -u tom
whoami  # Should say "tom"

# Try a restricted command as intern
sudo -i -u alex
rm -rf /  # Should FAIL 😊 (and that's good!)
```

---

## 🧠 Quick Memory Tricks (For Teaching!)

| Concept | Mnemonic |
|---------|----------|
| `useradd -m` | **M** = **M**ake home folder 🏠 |
| `usermod -aG` | **A**ppend to **G**roups (never forget the `-a`!) |
| `/etc/passwd` | **P**ublic **A**ccount **S**ummary **S**heet |
| `/etc/shadow` | **S**uper **H**idden **A**nd **D**angerous (passwords!) |
| `chage` | **CH**ange **AGE** of password ⏰ |
| `sudo` | **S**uper **U**ser **DO** — "Please let me do this as admin" |

---

## 🎓 How to Explain This to Others (30-Second Version)

> "Linux is like a secure apartment building.  
> - **Users** are residents with keys (accounts).  
> - **Groups** are families or teams that share access.  
> - **Commands** like `useradd` are like the front desk signing people in.  
> - **Files** like `/etc/passwd` are the tenant list; `/etc/shadow` is the locked key cabinet.  
> - **Password policies** are the building's security rules.  
> - And **privilege levels**? That's who gets the master key (root) vs. just their apartment key!"

---

## ✅ Quick Reference Cheat Sheet

```bash
# 👤 USER MANAGEMENT
useradd -m -s /bin/bash username     # Create user with home + bash
userdel -r username                  # Delete user + home folder
usermod -aG groupname username       # Add user to group (SAFE way!)
passwd username                      # Set/change password

# 👥 GROUP MANAGEMENT
groupadd groupname                   # Create group
gpasswd -a username groupname        # Add user to group
groups username                      # See user's groups

# 🔍 VIEW INFO
id username                          # Show UID, GID, groups
grep username /etc/passwd           # View user entry
chage -l username                   # View password policy for user

# 🔐 PASSWORD POLICY
chage -M 90 -W 7 username           # Expire in 90 days, warn 7 days before
passwd -l username                  # Lock account
passwd -u username                  # Unlock account
```

---

## 🚀 Bonus: Real-World Scenario to Practice

**Scenario**: You're setting up a startup's Linux server.

1. Create `ceo`, `dev_lead`, `intern` users
2. Make groups: `executives`, `engineering`, `readonly`
3. Give `ceo` sudo access, `dev_lead` access to `/var/www`, `intern` read-only
4. Enforce: passwords expire in 60 days, min 10 chars

> Try it in a VM or Docker container — safe sandbox for learning! 🧪

---